In [ ]:
import os  # OS 경로/폴더 생성 등 시스템 기능 사용
import torch  # PyTorch 메인 패키지
import torch.nn as nn  # 신경망 모듈(레이어, 손실 등)
from tqdm import tqdm  # 진행률 표시 바
from torchvision import models, transforms  # 사전학습 모델/이미지 변환
import numpy as np  # 수치 계산
from torch.amp import autocast, GradScaler  # 혼합정밀 자동 캐스트/스케일러
import math  # 수학 유틸
import time  # 시간 측정
from typing import Optional, Tuple, Dict  # 타입 힌트
import torch.nn.functional as F  # 함수형 API (loss/activation 등)
from sklearn.metrics import f1_score, classification_report, confusion_matrix, balanced_accuracy_score, top_k_accuracy_score  # 평가 지표
from datasets import load_from_disk  # HF dataset 디스크 로드(현재 코드에선 미사용)
from torch.autograd import Variable  # 오토그라드 Variable(현 PyTorch에선 텐서와 동일, 미사용)
from collections import Counter  # 라벨 카운트
import matplotlib.pyplot as plt

import json, os  # JSON 입출력/OS 유틸(중복 import but harmless)
from src.utils.labels import save_label_names
from src.utils.paths import build_fold_checkpoint_path, ensure_output_directory
from src.utils.runtime import resolve_device
from src.transforms.classification import build_finetune_transform, build_train_transform, build_validation_transform
from src.transforms.mixup import mixup_data
from src.datasets.fruit_freshness import FruitHFDataset, load_fruit_freshness_dataset
from src.datasets.folds import iter_stratified_folds, select_fold_datasets
from src.datasets.loaders import build_fold_dataloaders, build_holdout_dataloader
from src.models.factory import build_cmt_classifier
from src.losses.focal import FocalLoss, build_class_balanced_alpha
from src.losses.mixup import mixup_criterion
from src.engine.checkpoint import load_model_state, save_model_state
from src.engine.ema import ModelEma
from src.engine.optimization import build_optimizer, build_scheduler

In [ ]:
device = resolve_device()  # 사용 디바이스 선택(CUDA 우선)

train_transform = build_train_transform()
val_transform = build_validation_transform()




In [ ]:
# ------------------------------------------------------------
def load_fold_models(num_folds, num_classes, device, ckpt_dir):
    models = []
    for fold in range(1, num_folds + 1):
        m = build_cmt_classifier(num_classes).to(device)
        path = build_fold_checkpoint_path(ckpt_dir, fold)
        load_model_state(m, path, map_location=device)
        m.eval()
        models.append(m)
    return models

@torch.inference_mode()
def ensemble_logits(models, x):
    # logits 평균(softmax 전에 평균내는 방식)
    logits_sum = 0
    for m in models:
        logits_sum = logits_sum + m(x)
    return logits_sum / len(models)
@torch.inference_mode()
def ensemble_logits_tta_hflip(models, x):
    # x: (N,C,H,W)
    x_flip = torch.flip(x, dims=[3])  # width 축 flip
    logits = ensemble_logits(models, x)
    logits_flip = ensemble_logits(models, x_flip)
    return (logits + logits_flip) / 2


In [ ]:
# 메인
def main():  # 전체 파이프라인 실행 함수
    device = resolve_device()
    print("device:", device)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))

    final_dataset = load_fruit_freshness_dataset()
    names = final_dataset["train"].features["label"].names
    save_dir = ensure_output_directory("C:/Users/user/Desktop/deep/model_data")
    save_label_names(names, save_dir)

    num_classes = len(final_dataset["train"].features["label"].names)

    # ----- class-balanced alpha (FocalLoss용) -----
    train_labels = [int(x) for x in final_dataset["train"]["label"]]
    counts = Counter(train_labels)
    class_counts = [counts[i] for i in range(num_classes)]
    beta = 0.999
    alpha = build_class_balanced_alpha(class_counts, beta, num_classes)
    print("alpha:", alpha.tolist())


    # ================= [설정 변경] =================
    EPOCHS = 120
    # 마지막 5 epoch는 파인튜닝 (Cool-down)
    FINETUNE_EPOCHS = 20
    
    BATCH_SIZE = 192
    K = 3
    
    # Mixup 설정
    MIXUP_ALPHA = 0.8  # Mixup 강도 (0.08 -> 0.8로 약간 높임, 데이터가 적으면 강한게 좋음)
    MIXUP_P     = 0.5
    
    # 학습률
    LR_CNN = 5e-5
    LR_TRANS = 1e-4
    WEIGHT_DECAY = 1e-4
    
    # [전략] EMA Decay 설정
    EMA_DECAY = 0.999  
    
    USE_CE_LS = True
    LABEL_SMOOTHING = 0.01

    # [전략] 파인튜닝용 약한 증강 (ResizeCrop + Flip만 하고, ColorJitter/Erasing 제거)
    ft_transform = build_finetune_transform()
    # ===============================================


    fold_accs = []
    start_time = time.time()
    histories = []
    
    for fold, (train_idx, val_idx) in enumerate(iter_stratified_folds(final_dataset["train"], n_splits=K, shuffle=True, random_state=42), 1):
        best_acc_fold = 0.0
        print(f"\n================ Fold {fold}/{K} 시작 ================")
        fold_start = time.time()
            
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        train_split, val_split = select_fold_datasets(final_dataset["train"], train_idx, val_idx)

        # 데이터셋 생성
        train_ds = FruitHFDataset(train_split, transform=train_transform)
        val_ds   = FruitHFDataset(val_split,  transform=val_transform)

        train_loader, val_loader = build_fold_dataloaders(train_ds, val_ds, BATCH_SIZE)
        
        torch.backends.cudnn.benchmark = True

        # --- 모델 초기화 ---
        model = build_cmt_classifier(num_classes).to(device)
        
        # [전략] EMA 모델 초기화
        ema = ModelEma(model, decay=EMA_DECAY, device=device)

        if USE_CE_LS:
            criterion = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
        else:
            criterion = FocalLoss(alpha=alpha.to(device), gamma=2.0).to(device)

        optimizer = build_optimizer(
            model,
            lr_cnn=LR_CNN,
            lr_trans=LR_TRANS,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = build_scheduler(optimizer, t_max=EPOCHS)
        scaler = GradScaler()

        val_acc_list = []
        val_loss_list = []
        val_f1_list   = []

        for epoch in range(1, EPOCHS+1):
            epoch_start = time.time()
            
            # ================= [전략 1] 파인튜닝 (Cool-down) 체크 =================
            is_finetuning = (epoch > EPOCHS - FINETUNE_EPOCHS)
            
            if is_finetuning:
                print(f"▶ Fold {fold} | Epoch {epoch} [Fine-tuning Mode! Mixup OFF, Weak Aug]")
                # 1. 증강 교체 (강한 증강 -> 약한 증강)
                train_ds.tf = ft_transform 
                # 2. Mixup OFF (아래 루프에서 처리)
            else:
                print(f"\n▶ Fold {fold} | Epoch {epoch}/{EPOCHS}")
            # ======================================================================

            # ---- [Train] ----
            model.train()
            total, correct, loss_sum = 0, 0, 0.0
            pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch} [Train]", ncols=100)

            for x, y in pbar:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                # [전략] 파인튜닝 때는 Mixup 끄기
                if is_finetuning:
                    do_mix = False
                else:
                    do_mix = (np.random.rand() < MIXUP_P)

                if do_mix:
                    x_in, y_a, y_b, lam = mixup_data(x, y, alpha=MIXUP_ALPHA)
                else:
                    x_in, y_a, y_b, lam = x, y, y, 1.0

                with autocast("cuda"):
                    out = model(x_in)
                    if do_mix:
                        loss = mixup_criterion(criterion, out, y_a, y_b, lam)
                    else:
                        loss = criterion(out, y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                # [전략 2] EMA 업데이트 (학습 스텝마다)
                ema.update(model)

                bs = x.size(0)
                loss_sum += loss.item() * bs
                pred = out.argmax(1)
                if do_mix:
                    correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
                else:
                    correct += (pred == y).sum().item()
                total += bs

            tr_acc  = correct / max(1, total)
            tr_loss = loss_sum / max(1, total)
            print(f"Train ▶ acc: {tr_acc:.4f} | loss: {tr_loss:.4f}")

            # ---- [Validation] ----
            # [전략] 검증 시 EMA 모델 사용 (성능이 더 안정적임)
            # 주의: EMA 모델은 eval 모드이므로 model.eval() 불필요하지만 명시적으로 둠
            val_model = ema.module 
            val_model.eval()
            
            v_total, v_correct, v_loss_sum = 0, 0, 0.0
            all_preds, all_labels, all_logits = [], [], []

            with torch.inference_mode():
                for x, y in tqdm(val_loader, desc=f"Fold {fold} Epoch {epoch} [Val]", ncols=100):
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)

                    with autocast("cuda"):
                        # [전략 3] TTA (Horizontal Flip) 적용 - 검증부터 적용해서 성능 확인
                        out  = (val_model(x) + val_model(torch.flip(x, dims=[3]))) / 2
                        v_loss = criterion(out, y)

                    bs = x.size(0)
                    v_loss_sum += v_loss.item() * bs
                    preds = out.argmax(1)
                    v_correct += (preds == y).sum().item()
                    v_total   += bs

                    all_preds.extend(preds.detach().cpu().numpy())
                    all_labels.extend(y.detach().cpu().numpy())
                    all_logits.append(out.detach().cpu().numpy())

            va_acc  = v_correct / max(1, v_total)
            va_loss = v_loss_sum / max(1, v_total)
            
            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(va_loss)
            history["val_acc"].append(va_acc)
            
            all_logits = np.concatenate(all_logits, axis=0)
            va_f1   = f1_score(all_labels, all_preds, average="macro")
            va_bal  = balanced_accuracy_score(all_labels, all_preds)

            try:
                va_top2 = top_k_accuracy_score(all_labels, all_logits, k=2, labels=np.arange(all_logits.shape[1]))
                va_top3 = top_k_accuracy_score(all_labels, all_logits, k=3, labels=np.arange(all_logits.shape[1]))
            except:
                va_top2 = va_top3 = None

            val_acc_list.append(va_acc)
            val_loss_list.append(va_loss)
            val_f1_list.append(va_f1)
            print(f"Val (EMA) ▶ acc: {va_acc:.4f} | f1: {va_f1:.4f} | loss: {va_loss:.4f}")
            
            # 저장
            if va_acc > best_acc_fold + 1e-6:
                best_acc_fold = va_acc
                save_path = build_fold_checkpoint_path(save_dir, fold)
                # [전략] EMA 모델의 가중치를 저장
                save_model_state(ema.module, save_path)
                print(f"New best model (EMA) saved! (fold={fold}, acc={best_acc_fold:.4f})")

            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch} 완료 (소요시간: {epoch_time:.2f}초)")

            scheduler.step()

        # --- Fold 종료 ---
        fold_time = time.time() - fold_start
        histories.append(history)
        print(f"✅ Fold {fold} 완료! (소요시간: {fold_time/60:.2f}분)")
        fold_accs.append(val_acc_list)

    # --- 전체 요약 ---
    ckpt_dir = save_dir
    models = load_fold_models(K, num_classes, device, ckpt_dir)

    test_ds = FruitHFDataset(final_dataset["test"], transform=val_transform)
    test_loader = build_holdout_dataloader(test_ds, BATCH_SIZE)

    t_total = t_correct = 0
    print("\n[최종 평가] Holdout Test Set (Ensemble + TTA)")
    
    # [전략] TTA + 앙상블 적용
    for x, y in tqdm(test_loader, ncols=100):
        x = x.to(device)
        y = y.to(device)
        with autocast("cuda"):
            logits = ensemble_logits_tta_hflip(models, x)
        pred = logits.argmax(1)
        t_correct += (pred == y).sum().item()
        t_total += y.size(0)

    print("Final Holdout Acc:", t_correct / t_total)
    
    total_time = time.time() - start_time
    print(f"\n================ 학습종료 총 소요시간: {total_time/60:.2f}분 ================")

    # 그래프 그리기
    epochs = range(1, EPOCHS + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    # 마지막 Fold의 history만 그림
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"],   label="val_loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss (Last Fold)"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"],   label="val_acc")
    plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.title("Accuracy (Last Fold)"); plt.legend()
    plt.tight_layout()
    plt.show()

    save_model_state(model, "last_model_weights.pt")

if __name__ == "__main__":
    main()